### Import modules

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow import keras
# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


2025-12-22 11:09:08.419645: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Load Data

In [13]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
# data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

data = data.drop(columns=['file'])
data['pattern'] = data['pattern'].fillna('None Type')
data.head()

,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,...,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,dim_768,pattern
0,0.008586,-0.000224,-0.073952,0.040551,0.041076,0.051949,0.053746,0.020028,0.009389,0.006378,...,0.031212,-0.029720,0.062391,0.007221,0.005423,-0.022622,-0.054182,0.068976,-0.041483,Advanced LLM Prompting
1,0.025723,0.010738,-0.067647,0.041301,0.041939,0.043754,0.024987,0.018802,0.020988,-0.032886,...,0.035231,-0.000288,0.062254,0.007914,0.032241,-0.018615,-0.059511,0.072126,-0.037370,Advanced LLM Prompting
2,-0.038119,0.056748,-0.046379,0.022291,0.039093,0.036978,0.027474,0.021547,-0.009878,-0.018463,...,0.066070,-0.008232,0.025106,0.024327,-0.002147,-0.049329,-0.048281,0.095675,-0.039835,Advanced LLM Prompting
3,-0.007249,0.027223,-0.050162,0.021737,0.056431,0.058930,0.055523,-0.044478,-0.016022,-0.008388,...,0.005574,-0.011538,0.017119,0.023677,-0.014072,-0.029506,-0.076149,0.091361,-0.017112,Advanced LLM Prompting
4,0.002775,-0.007918,-0.072574,0.027786,0.067952,0.040754,0.045165,-0.037903,-0.023210,0.013282,...,0.037829,0.023176,0.052465,-0.020104,0.026510,-0.020693,-0.050852,0.023219,-0.009296,Advanced LLM Prompting


### Configure

In [5]:
EVALUATING_ENABLED = False
TEMP_TEST_SPLIT = True

In [14]:
if TEMP_TEST_SPLIT:
    temp_train_data,temp_test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data['pattern'])
    data = temp_train_data
    temp_test_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_test_data.csv', index=False)

### NN Architecture

In [15]:
TARGET_COLUMN = "pattern"

ARTIFACT_DIR = Path("../models/pattern_nn_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "pattern_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
ENCODER_PATH = ARTIFACT_DIR / "label_encoder.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

In [16]:
numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_features:
    raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

X = data[numeric_features].fillna(0.0).values
y = data[TARGET_COLUMN].astype(str).values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

In [17]:
X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
    X,
    y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded,
)
X_val, X_test, y_val_enc, y_test_enc = train_test_split(
    X_temp,
    y_temp_enc,
    test_size=0.5,
    random_state=42,
    stratify=y_temp_enc,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)


if not EVALUATING_ENABLED:
    # X_train = np.vstack([X_train, X_val])
    # y_train = np.vstack([y_train, y_val])
    X_val = np.vstack([X_val, X_test])
    y_val = np.vstack([y_val, y_test])

def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            # Dense(num_classes, activation="softmax"),
            Dense(num_classes, activation=None),
        ]
    )

@keras.utils.register_keras_serializable()
class TemperatureScaling(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.temperature = tf.Variable(
            initial_value=1.0,
            trainable=True,
            dtype=tf.float32,
            constraint=lambda t: tf.clip_by_value(t, 1e-6, 100.0),
        )

    def call(self, logits):
        return logits / self.temperature
    def get_config(self):
        return super().get_config()

def build_calibrated_model(base_model: tf.keras.Model) -> tf.keras.Model:
    base_model.trainable = False

    inputs = tf.keras.Input(shape=base_model.input_shape[1:])
    logits = base_model(inputs)

    scaled_logits = TemperatureScaling()(logits)
    outputs = tf.keras.layers.Softmax()(scaled_logits)

    return tf.keras.Model(inputs, outputs)

import numpy as np

def expected_calibration_error(
    probs: np.ndarray,
    y_true: np.ndarray,
    n_bins: int = 15,
) -> float:
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == y_true).astype(float)

    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(y_true)

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]

        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        bin_size = np.sum(in_bin)

        if bin_size > 0:
            bin_accuracy = np.mean(accuracies[in_bin])
            bin_confidence = np.mean(confidences[in_bin])

            ece += (bin_size / N) * abs(bin_accuracy - bin_confidence)

    return ece


model = build_classifier(X_train.shape[1], num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

calibrated_model = build_calibrated_model(model)

calibrated_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)


calibrated_model.fit(
    X_val,
    y_val,
    epochs=50,
    batch_size=256,
    verbose=0,
)


if not EVALUATING_ENABLED:
    # Persist artifacts for downstream inference pipelines
    calibrated_model.save(MODEL_PATH, include_optimizer=True)
    joblib.dump(scaler, SCALER_PATH)
    joblib.dump(label_encoder, ENCODER_PATH)
    metadata = {
        "target_column": TARGET_COLUMN,
        "numeric_features": numeric_features,
        "num_classes": num_classes,
        "label_classes": label_encoder.classes_.tolist(),
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2))
    print(f"Saved model to {MODEL_PATH}")
    print(f"Saved scaler to {SCALER_PATH}")
    print(f"Saved label encoder to {ENCODER_PATH}")
    print(f"Saved metadata to {METADATA_PATH}")
else:
    print("\nEvaluation enabled; not saving model artifacts.")

    test_loss, test_acc, test_top3 = calibrated_model.evaluate(X_test, y_test, verbose=0)
    print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

    y_pred = calibrated_model.predict(X_test)
    y_pred_labels = y_pred.argmax(axis=1)
    report = classification_report(
        y_test_enc,
        y_pred_labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    probs = calibrated_model.predict(X_test)
    ece = expected_calibration_error(probs, y_test_enc, n_bins=15)

    print(f"ECE: {ece:.4f}")


    print("\nKey metrics:")
    display(summary)
    print("\nTop classes by support:")
    display(class_breakdown)

    raw_preds = model.predict(X_test).argmax(axis=1)
    cal_preds = calibrated_model.predict(X_test).argmax(axis=1)

    print("Accuracy identical:", np.all(raw_preds == cal_preds))

2025-12-22 11:14:18.617018: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Epoch 1/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 104ms/step - accuracy: 0.2496 - loss: 3.0348 - top3_acc: 0.4056 - val_accuracy: 0.4646 - val_loss: 2.1270 - val_top3_acc: 0.6195 - learning_rate: 0.0010
Epoch 2/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.5755 - loss: 1.7144 - top3_acc: 0.7438 - val_accuracy: 0.5832 - val_loss: 1.6492 - val_top3_acc: 0.7686 - learning_rate: 0.0010
Epoch 3/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.6716 - loss: 1.3362 - top3_acc: 0.8407 - val_accuracy: 0.6635 - val_loss: 1.4152 - val_top3_acc: 0.8413 - learning_rate: 0.0010
Epoch 4/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.7463 - loss: 1.0424 - top3_acc: 0.9130 - val_accuracy: 0.6922 - val_loss: 1.2846 - val_top3_acc: 0.8642 - learning_rate: 0.0010
Epoch 5/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.8030 - loss: 0.8361 - top3_acc: 0.9491 - val_accuracy: 0.7094 - val_loss: 1.1994 - val_top3_acc: 0.8776 - learning_rate: 0.0010
Epoch 6/200
20/20 ━━━━━━━━━━━

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json


In [18]:
INFERENCE_SOURCE_PATH = Path(
    "/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding.csv"
)

inference_df = pd.read_csv(INFERENCE_SOURCE_PATH)
numeric_columns = inference_df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_columns:
    raise ValueError("No numeric columns detected in the inference dataset.")

context_columns = [col for col in inference_df.columns if col not in numeric_columns]
feature_matrix = inference_df[numeric_columns].fillna(0.0).values
# embeddings = np.atleast_2d(feature_matrix)
scaled = scaler.transform(feature_matrix)
probs = calibrated_model.predict(scaled)
pd.DataFrame(probs, columns=label_encoder.classes_)


77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


,Advanced LLM Prompting,Classical Models,Cross-lingual LLM Prompting,Enhanced User Intent Comprehension with LLMs,Explainable AI (XAI) Techniques,Integrating External Knlowladge with LLM,LLM Agent Training & Alignment,LLM Code Execution for Precision,LLM Context Management,LLM KV Cache Optimization,...,"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",LLMs for Recommender Systems,Model Abstraction Pattern,Modular LLM Agent Architectures,None Type,Preprocessing Text and Numerical Data,"Reliable, Transparent, & Augmented LLMs",Retrieval Augmented Generation(RAG) Optimization for LLMs,Structured Output & Formatting for LLMs,Tool Use for LLMs
0,0.000049,0.000197,0.000074,4.256602e-06,0.000052,0.000017,0.001512,0.000041,1.666985e-05,0.000084,...,0.000021,0.000716,0.000065,0.000045,0.583431,0.002982,0.000187,0.000654,0.000025,0.000171
1,0.000018,0.000059,0.000003,1.495070e-07,0.000017,0.000001,0.000080,0.000007,8.238959e-07,0.000006,...,0.000005,0.000047,0.000004,0.000005,0.997786,0.000556,0.000012,0.000065,0.000002,0.000017
2,0.000014,0.000125,0.000067,2.481706e-06,0.000006,0.000014,0.000289,0.000015,5.403857e-06,0.000015,...,0.000003,0.000143,0.000023,0.000007,0.054176,0.001307,0.000037,0.000245,0.000015,0.000112
3,0.000611,0.044224,0.000794,5.020220e-04,0.002526,0.001456,0.000968,0.000471,3.276634e-04,0.001008,...,0.000433,0.002097,0.000272,0.000193,0.818955,0.096635,0.001320,0.000715,0.011556,0.003463
4,0.008935,0.003521,0.001628,6.119899e-03,0.004325,0.009522,0.032963,0.001436,4.906133e-02,0.003894,...,0.106304,0.005955,0.002984,0.005689,0.101795,0.198470,0.388239,0.011234,0.002714,0.032303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2437,0.008755,0.000424,0.000272,1.268297e-04,0.003585,0.004802,0.002163,0.000154,9.490641e-05,0.008205,...,0.000156,0.001901,0.004868,0.000131,0.057910,0.005569,0.081695,0.004348,0.000479,0.005825
2438,0.006003,0.000008,0.000143,7.118150e-04,0.000627,0.001345,0.000664,0.000166,1.181363e-04,0.001516,...,0.000084,0.000765,0.000514,0.000677,0.480320,0.001630,0.144876,0.003080,0.000032,0.000016
2439,0.000649,0.037197,0.000024,3.409667e-04,0.001294,0.034840,0.000285,0.000831,1.908717e-04,0.000414,...,0.000296,0.000034,0.003886,0.048226,0.725189,0.019698,0.000200,0.002137,0.002409,0.119777
2440,0.006325,0.003680,0.007896,1.939130e-02,0.000869,0.003174,0.000609,0.000189,7.393945e-05,0.038624,...,0.000251,0.000585,0.001489,0.001135,0.807293,0.005004,0.000131,0.068129,0.000090,0.005790


In [ ]:
pd.DataFrame(feature_matrix)

### Logistic Regression classifier

In [19]:
# Logistic regression stage intentionally skipped per latest workflow requirements.
# The end-to-end classifier now relies solely on the neural network above.
numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_features:
    raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

X = data[numeric_features].fillna(0.0).values
y = data[TARGET_COLUMN].astype(str).values
scaler = StandardScaler()
X = scaler.fit_transform(X)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.25, random_state=42,stratify=y_encoded
)

In [20]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000,n_jobs=-1)

if EVALUATING_ENABLED:
    logreg.fit(X_train, y_train)
    y_pred = logreg.predict(X_test)
    report = classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    display(summary)
else:
    logreg.fit(X, y)
    # Save model
    ARTIFACT_DIR = Path("../models/pattern_logreg_classifier").resolve()
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(logreg, ARTIFACT_DIR / "logistic_regression_model.joblib")
    joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")
    joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")
    print(f"Saved logistic regression model and artifacts to {ARTIFACT_DIR}")


Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


### SVC

In [ ]:
#import svc from sklearn.svm import SVC
from sklearn.svm import SVC
svc = SVC(probability=True)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
report = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
class_breakdown = (
    report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
)   
summary

In [ ]:
ARTIFACT_DIR = Path("../models/pattern_svc_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "svc_model.joblib"
joblib.dump(svc, MODEL_PATH)
joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")